In [ ]:
code = '''"""
Zonal TFT — refit through 2023 + heat-memory features (fx).
Two changes at once: training window extended, three fx features added.
Base to beat: 3.20% lead-matched MAPE.
"""
from pathlib import Path
import lightning.pytorch as pl
import pandas as pd
import torch
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

FEATURES = Path("zonal_features_fx.parquet")
CKPT_DIR = Path("checkpoints_zonal")
TRAIN_START = "2015-07-01 04:00"
VAL_START   = "2024-01-01 05:00"
TEST_START  = "2024-01-23 12:00"
ENCODER_LEN, DECODER_LEN = 168, 120
BATCH = 128; MAX_EPOCHS = 2; SEED = 42
NUM_WORKERS = 4

WEATHER = ["temperature_2m","apparent_temperature","relative_humidity_2m",
           "wind_speed_10m","shortwave_radiation","cloud_cover","temp_vshape"]
FX = ["fx_app_roll72","fx_cdh24","fx_hot_streak_day"]
UNKNOWN_REALS = ["demand","demand_lag24","demand_lag168",
                 "demand_roll24_mean","demand_roll168_mean","demand_roll24_std"]
KNOWN_REALS = WEATHER + FX + ["time_idx"]
KNOWN_CATS = ["hour","day_of_week","month","is_weekend","is_holiday"]

def main():
    pl.seed_everything(SEED)
    torch.set_float32_matmul_precision("high")

    df = pd.read_parquet(FEATURES)
    df["utc"] = pd.to_datetime(df["utc"])
    for c in KNOWN_CATS: df[c] = df[c].astype(str).astype("category")
    df["zone"] = df["zone"].astype(str)
    df = df[df["utc"] >= TRAIN_START].copy()
    df["time_idx"] = df["time_idx"] - df["time_idx"].min()

    missing = [c for c in KNOWN_REALS if c not in df.columns]
    if missing:
        raise SystemExit(f"missing columns: {missing}")

    val_idx = int(df.loc[df["utc"] >= VAL_START, "time_idx"].min())
    test_idx = int(df.loc[df["utc"] >= TEST_START, "time_idx"].min())
    print(f"boundaries - val:{val_idx} test:{test_idx} max:{int(df[\\'time_idx\\'].max())}", flush=True)
    print(f"train rows: {len(df[df[\\'time_idx\\'] < val_idx]):,}", flush=True)

    train_df = df[df["time_idx"] < val_idx]
    training_ds = TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["zone"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )
    validation_ds = TimeSeriesDataSet.from_dataset(
        training_ds, df[df["time_idx"] < test_idx],
        min_prediction_idx=val_idx, stop_randomization=True)

    train_dl = training_ds.to_dataloader(train=True, batch_size=BATCH,
                                         num_workers=NUM_WORKERS, persistent_workers=True)
    val_dl = validation_ds.to_dataloader(train=False, batch_size=BATCH,
                                         num_workers=NUM_WORKERS, persistent_workers=True)

    model = TemporalFusionTransformer.from_dataset(
        training_ds,
        hidden_size=64,
        attention_head_size=4,
        dropout=0.1,
        hidden_continuous_size=32,
        loss=QuantileLoss(),
        learning_rate=3e-4,
        reduce_on_plateau_patience=2,
        log_interval=100,
    )
    print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M", flush=True)

    CKPT_DIR.mkdir(exist_ok=True)
    ckpt_cb = ModelCheckpoint(dirpath=CKPT_DIR, filename="zonal_tft_refit2023_fx_best",
                              monitor="val_loss", mode="min", save_top_k=1,
                              save_last=True)
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, accelerator="auto",
        gradient_clip_val=0.1,
        callbacks=[ckpt_cb, LearningRateMonitor(logging_interval="epoch")],
        enable_progress_bar=False,
        log_every_n_steps=200,
    )

    resume = CKPT_DIR / "last.ckpt"
    resume_path = str(resume) if resume.exists() else None
    print(f"resuming from: {resume_path}", flush=True)

    trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl,
                ckpt_path=resume_path, weights_only=False)

    print("Best checkpoint:", ckpt_cb.best_model_path, flush=True)
    print("Best val_loss:  ", float(ckpt_cb.best_model_score), flush=True)

if __name__ == "__main__":
    main()
'''

path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_train_refit2023_fx.py"
with open(path, "w") as f:
    f.write(code)
print("written:", path)

In [1]:
import subprocess, sys, os, time

ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

stale = f"{ROOT}/checkpoints_zonal/last.ckpt"
if os.path.exists(stale):
    os.remove(stale)
    print("removed stale last.ckpt")

print("GPU:")
print(subprocess.run(["nvidia-smi","--query-compute-apps=pid,used_memory","--format=csv"],
                     capture_output=True, text=True).stdout)
print("already training:", subprocess.run(["pgrep","-af","05_train_refit2023_fx"],
                                          capture_output=True, text=True).stdout.strip() or "none")

log = f"{ROOT}/train_refit2023_fx.log"
with open(log, "w") as f:
    subprocess.Popen([sys.executable, "05_train_refit2023_fx.py"],
                     stdout=f, stderr=subprocess.STDOUT, start_new_session=True, cwd=ROOT)
print("\nlaunched")
time.sleep(90)
print(subprocess.run(["tail","-8",log], capture_output=True, text=True).stdout)

removed stale last.ckpt
GPU:
pid, used_gpu_memory [MiB]
5404, 250 MiB

already training: none

launched
/opt/app-root/bin/python3: can't open file '/opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_train_refit2023_fx.py': [Errno 2] No such file or directory



In [2]:
!tail -12 /opt/app-root/src/Forecasting-Energy-Demand/Sangar/train_refit2023_fx.log

/opt/app-root/bin/python3: can't open file '/opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_train_refit2023_fx.py': [Errno 2] No such file or directory


In [3]:
import subprocess, os
ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"
print("cwd:", os.getcwd())
print("\n--- any refit files anywhere under the repo ---")
print(subprocess.run(["find","/opt/app-root/src/Forecasting-Energy-Demand",
                      "-name","*refit*"], capture_output=True, text=True).stdout or "none found")
print("--- python files in Sangar ---")
print(subprocess.run(["ls","-lt",ROOT], capture_output=True, text=True).stdout[:1500])

cwd: /opt/app-root/src/Forecasting-Energy-Demand/Sangar

--- any refit files anywhere under the repo ---
/opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_train_refit2023_fx.ipynb
/opt/app-root/src/Forecasting-Energy-Demand/Sangar/.ipynb_checkpoints/05_train_refit2023_fx-checkpoint.ipynb
/opt/app-root/src/Forecasting-Energy-Demand/Sangar/train_refit2023_fx.log

--- python files in Sangar ---
total 355340
-rw-r--r--.  1 1000950000 1000950000     8523 Jul 23 12:43 05_train_refit2023_fx.ipynb
-rw-r--r--.  1 1000950000 1000950000      158 Jul 23 12:41 train_refit2023_fx.log
drwxrwsr-x.  2 1000950000 1000950000     4096 Jul 23 12:41 checkpoints_zonal
-rw-r--r--.  1 1000950000 1000950000     3896 Jul 23 12:39 zonal_features_fx.parquet.ipynb
-rw-r--r--.  1 1000950000 1000950000 70790177 Jul 23 12:37 zonal_features_fx.parquet
-rw-r--r--.  1 1000950000 1000950000    10953 Jul 23 12:22 new_features_analysis.ipynb
-rw-r--r--.  1 1000950000 1000950000 68401017 Jul 23 12:10 zonal_features_dewpo

In [4]:
code = '''"""
Zonal TFT — refit through 2023 + heat-memory features (fx).
Base to beat: 3.20% lead-matched MAPE.
"""
from pathlib import Path
import lightning.pytorch as pl
import pandas as pd
import torch
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

FEATURES = Path("zonal_features_fx.parquet")
CKPT_DIR = Path("checkpoints_zonal")
TRAIN_START = "2015-07-01 04:00"
VAL_START   = "2024-01-01 05:00"
TEST_START  = "2024-01-23 12:00"
ENCODER_LEN, DECODER_LEN = 168, 120
BATCH = 128; MAX_EPOCHS = 2; SEED = 42
NUM_WORKERS = 4

WEATHER = ["temperature_2m","apparent_temperature","relative_humidity_2m",
           "wind_speed_10m","shortwave_radiation","cloud_cover","temp_vshape"]
FX = ["fx_app_roll72","fx_cdh24","fx_hot_streak_day"]
UNKNOWN_REALS = ["demand","demand_lag24","demand_lag168",
                 "demand_roll24_mean","demand_roll168_mean","demand_roll24_std"]
KNOWN_REALS = WEATHER + FX + ["time_idx"]
KNOWN_CATS = ["hour","day_of_week","month","is_weekend","is_holiday"]

def main():
    pl.seed_everything(SEED)
    torch.set_float32_matmul_precision("high")

    df = pd.read_parquet(FEATURES)
    df["utc"] = pd.to_datetime(df["utc"])
    for c in KNOWN_CATS: df[c] = df[c].astype(str).astype("category")
    df["zone"] = df["zone"].astype(str)
    df = df[df["utc"] >= TRAIN_START].copy()
    df["time_idx"] = df["time_idx"] - df["time_idx"].min()

    missing = [c for c in KNOWN_REALS if c not in df.columns]
    if missing:
        raise SystemExit("missing columns: " + str(missing))

    val_idx = int(df.loc[df["utc"] >= VAL_START, "time_idx"].min())
    test_idx = int(df.loc[df["utc"] >= TEST_START, "time_idx"].min())
    max_idx = int(df["time_idx"].max())
    n_train = len(df[df["time_idx"] < val_idx])
    print("boundaries - val:%d test:%d max:%d" % (val_idx, test_idx, max_idx), flush=True)
    print("train rows: %d" % n_train, flush=True)

    train_df = df[df["time_idx"] < val_idx]
    training_ds = TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="demand", group_ids=["zone"],
        max_encoder_length=ENCODER_LEN, max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        static_categoricals=["zone"],
        target_normalizer=GroupNormalizer(groups=["zone"]),
        add_relative_time_idx=True, add_target_scales=True,
        allow_missing_timesteps=False,
    )
    validation_ds = TimeSeriesDataSet.from_dataset(
        training_ds, df[df["time_idx"] < test_idx],
        min_prediction_idx=val_idx, stop_randomization=True)

    train_dl = training_ds.to_dataloader(train=True, batch_size=BATCH,
                                         num_workers=NUM_WORKERS, persistent_workers=True)
    val_dl = validation_ds.to_dataloader(train=False, batch_size=BATCH,
                                         num_workers=NUM_WORKERS, persistent_workers=True)

    model = TemporalFusionTransformer.from_dataset(
        training_ds,
        hidden_size=64,
        attention_head_size=4,
        dropout=0.1,
        hidden_continuous_size=32,
        loss=QuantileLoss(),
        learning_rate=3e-4,
        reduce_on_plateau_patience=2,
        log_interval=100,
    )
    print("Parameters: %.2fM" % (sum(p.numel() for p in model.parameters())/1e6), flush=True)

    CKPT_DIR.mkdir(exist_ok=True)
    ckpt_cb = ModelCheckpoint(dirpath=CKPT_DIR, filename="zonal_tft_refit2023_fx_best",
                              monitor="val_loss", mode="min", save_top_k=1,
                              save_last=True)
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, accelerator="auto",
        gradient_clip_val=0.1,
        callbacks=[ckpt_cb, LearningRateMonitor(logging_interval="epoch")],
        enable_progress_bar=False,
        log_every_n_steps=200,
    )

    resume = CKPT_DIR / "last.ckpt"
    resume_path = str(resume) if resume.exists() else None
    print("resuming from: %s" % resume_path, flush=True)

    trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl,
                ckpt_path=resume_path, weights_only=False)

    print("Best checkpoint:", ckpt_cb.best_model_path, flush=True)
    print("Best val_loss:  ", float(ckpt_cb.best_model_score), flush=True)

if __name__ == "__main__":
    main()
'''

import os
path = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_train_refit2023_fx.py"
with open(path, "w") as f:
    f.write(code)
print("written:", path, "|", os.path.getsize(path), "bytes")

written: /opt/app-root/src/Forecasting-Energy-Demand/Sangar/05_train_refit2023_fx.py | 4532 bytes


In [9]:
import subprocess, os
ROOT = "/opt/app-root/src/Forecasting-Energy-Demand/Sangar"

print("process:", subprocess.run(["pgrep","-f","05_train_refit2023_fx"],
      capture_output=True, text=True).stdout.strip() or "DEAD")
print("\n--- last log lines ---")
print(subprocess.run(["tail","-8", f"{ROOT}/train_refit2023_fx.log"],
      capture_output=True, text=True).stdout)
print("--- checkpoints ---")
print(subprocess.run(["ls","-lt", f"{ROOT}/checkpoints_zonal/"],
      capture_output=True, text=True).stdout)

process: DEAD

--- last log lines ---
Total params: 450 K                                                             
Total estimated model params size (MB): 1.802                                   
Modules in train mode: 679                                                      
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  
`Trainer.fit` stopped: `max_epochs=2` reached.
Best checkpoint: /opt/app-root/src/Forecasting-Energy-Demand/Sangar/checkpoints_zonal/zonal_tft_refit2023_fx_best.ckpt
Best val_loss:   23.411636352539062

--- checkpoints ---
total 33584
-rw-r--r--. 1 1000950000 1000950000 6173587 Jul 23 13:36 last.ckpt
-rw-r--r--. 1 1000950000 1000950000 6173523 Jul 23 13:36 zonal_tft_refit2023_fx_best.ckpt
-rw-r--r--. 1 1000950000 1000950000 5518514 Jul 22 13:34 zonal_tft_lr3e4_best-v1.ckpt
-rw-r--r--. 1 1000950000 1000950000 5503794 Jul 22 05:09 zonal_tft_lr3e4_best.c